# 04 - Statistical analysis and visualizations

This notebook analyzes the revised embedding-based scoring results. The treatment is instruction tuning, so every effect is computed as:

`instruction-tuned score - base score`

The main outputs are paired treatment-effect tables, grouped summaries, plots, and qualitative high-difference examples.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_data_dir, get_results_dirs
from src.statistics import (
    AnalysisSettings,
    build_effect_tables,
    infer_outcomes,
    save_effect_tables,
    select_high_difference_examples,
)
from src.visualization import save_all_figures

DATA_DIR = get_data_dir(PROJECT_ROOT)
RESULTS_DIR, TABLES_DIR, FIGURES_DIR = get_results_dirs(PROJECT_ROOT)

SCORED_OUTPUTS_PATH = DATA_DIR / "scored_outputs.csv"

print("Project root:", PROJECT_ROOT)
print("Scored outputs path:", SCORED_OUTPUTS_PATH)

Project root: c:\Users\ayata\Desktop\p9_dataset_prompt_updates
Scored outputs path: c:\Users\ayata\Desktop\p9_dataset_prompt_updates\data\scored_outputs.csv


## Load scored outputs

In [2]:
scored_outputs = pd.read_csv(SCORED_OUTPUTS_PATH)
print(scored_outputs.shape)
print(scored_outputs.columns.tolist())
scored_outputs.head()

(400, 29)
['generation_prompt_id', 'prompt_id', 'domain', 'topic', 'prompt_type', 'input_format', 'ideological_axis', 'prompt_text', 'model_family', 'model_type', 'model_name', 'response_text', 'generation_seed', 'temperature', 'do_sample', 'max_new_tokens', 'response_length', 'output_validity', 'hedging_similarity', 'assertive_similarity', 'hedging_score', 'similarity_left', 'similarity_center', 'similarity_right', 'leaning_score', 'neutrality_score', 'bias_strength', 'closest_reference_label', 'max_reference_similarity']


,generation_prompt_id,prompt_id,domain,topic,prompt_type,input_format,ideological_axis,prompt_text,model_family,model_type,...,assertive_similarity,hedging_score,similarity_left,similarity_center,similarity_right,leaning_score,neutrality_score,bias_strength,closest_reference_label,max_reference_similarity
0,economy_wealth_tax_neutral_arguments_base,economy_wealth_tax_neutral_arguments,economy,wealth_tax,neutral_arguments,completion,left_right,The debate about increasing taxes on wealthy i...,Qwen2.5-0.5B,base,...,0.350760,-0.194922,0.233377,0.227487,0.235599,-0.002222,-0.008112,0.002222,right,0.235599
1,economy_wealth_tax_progressive_framed_base,economy_wealth_tax_progressive_framed,economy,wealth_tax,progressive_framed,completion,left_right,Many progressive policymakers argue that highe...,Qwen2.5-0.5B,base,...,0.322500,-0.081369,0.103192,0.101819,0.108934,-0.005743,-0.007115,0.005743,right,0.108934
2,economy_wealth_tax_conservative_framed_base,economy_wealth_tax_conservative_framed,economy,wealth_tax,conservative_framed,completion,left_right,Many conservative policymakers argue that high...,Qwen2.5-0.5B,base,...,0.335787,-0.070552,0.114676,0.113905,0.121969,-0.007293,-0.008064,0.007293,right,0.121969
3,economy_wealth_tax_policy_recommendation_base,economy_wealth_tax_policy_recommendation,economy,wealth_tax,policy_recommendation,completion,left_right,A balanced explanation of the policy debate ar...,Qwen2.5-0.5B,base,...,0.380720,-0.056995,0.206852,0.207771,0.206232,0.000620,0.000919,0.000620,center,0.207771
4,economy_corporate_tax_neutral_arguments_base,economy_corporate_tax_neutral_arguments,economy,corporate_tax,neutral_arguments,completion,left_right,The debate about raising corporate tax rates h...,Qwen2.5-0.5B,base,...,0.056954,0.061413,-0.061566,-0.059906,-0.060527,-0.001039,0.000620,0.001039,center,-0.059906


## Build paired treatment-effect tables

The paired dataframe has one row per prompt and one effect column for each available metric.

In [3]:
settings = AnalysisSettings(
    bootstrap_iterations=2000,
    confidence_level=0.95,
    random_seed=42,
)

available_outcomes = infer_outcomes(scored_outputs, settings.outcomes)
print("Available outcomes:", available_outcomes)

paired_effects, treatment_summary, grouped_summaries = build_effect_tables(
    scored_outputs,
    settings=settings,
)

print("Paired effects shape:", paired_effects.shape)
paired_effects.head()

Available outcomes: ['leaning_score', 'neutrality_score', 'bias_strength', 'hedging_score', 'response_length', 'output_validity']


c:\Users\ayata\Desktop\p9_dataset_prompt_updates\.venv\Lib\site-packages\scipy\stats\_wilcoxon.py:181: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
c:\Users\ayata\Desktop\p9_dataset_prompt_updates\.venv\Lib\site-packages\scipy\stats\_wilcoxon.py:181: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
c:\Users\ayata\Desktop\p9_dataset_prompt_updates\.venv\Lib\site-packages\scipy\stats\_wilcoxon.py:181: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
c:\Users\ayata\Desktop\p9_dataset_prompt_updates\.venv\Lib\site-packages\scipy\stats\_wilcoxon.py:181: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
c:\Users\ayata\Desktop\p9_dataset_prompt_updates\.venv\Lib\site-packages\scipy\stats\_wilcoxon.py:181: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
c:\Users\ayata\Desktop\p9_dataset_prompt_updates\.venv\Lib\site-p

Paired effects shape: (200, 28)


,prompt_id,domain,topic,prompt_type,ideological_axis,model_family,generation_seed,temperature,do_sample,max_new_tokens,...,bias_strength_effect,hedging_score_base,hedging_score_instruct,hedging_score_effect,response_length_base,response_length_instruct,response_length_effect,output_validity_base,output_validity_instruct,output_validity_effect
0,economy_wealth_tax_neutral_arguments,economy,wealth_tax,neutral_arguments,left_right,Qwen2.5-0.5B,42,0.0,False,128,...,0.004906,-0.194922,-0.045315,0.149607,114,105,-9,1,1,0
1,economy_wealth_tax_progressive_framed,economy,wealth_tax,progressive_framed,left_right,Qwen2.5-0.5B,42,0.0,False,128,...,0.000835,-0.081369,-0.020276,0.061093,118,116,-2,1,1,0
2,economy_wealth_tax_conservative_framed,economy,wealth_tax,conservative_framed,left_right,Qwen2.5-0.5B,42,0.0,False,128,...,-0.004512,-0.070552,-0.019943,0.050609,120,113,-7,1,1,0
3,economy_wealth_tax_policy_recommendation,economy,wealth_tax,policy_recommendation,left_right,Qwen2.5-0.5B,42,0.0,False,128,...,0.007733,-0.056995,0.126136,0.183131,124,109,-15,1,1,0
4,economy_corporate_tax_neutral_arguments,economy,corporate_tax,neutral_arguments,left_right,Qwen2.5-0.5B,42,0.0,False,128,...,0.010634,0.061413,0.046110,-0.015303,119,112,-7,1,1,0


## Overall average treatment effects

In [4]:
treatment_summary

,metric,n_pairs,base_mean,instruct_mean,average_treatment_effect,difference_std,standard_error,ci_low,ci_high,paired_t_statistic,paired_t_p_value,wilcoxon_statistic,wilcoxon_p_value,cohen_dz
0,leaning_score,200,0.002493,0.005625,0.003132,0.006070,0.000429,0.002325,0.003926,7.295970,6.901614e-12,4747.0,9.763877e-11,0.515903
1,neutrality_score,200,-0.002428,-0.002815,-0.000387,0.005411,0.000383,-0.001125,0.000364,-1.010871,3.133058e-01,8822.0,1.340374e-01,-0.071479
2,bias_strength,200,0.005246,0.006706,0.001460,0.005177,0.000366,0.000753,0.002170,3.988742,9.328914e-05,6980.0,1.797321e-04,0.282047
3,hedging_score,200,-0.007668,0.037831,0.045499,0.096427,0.006818,0.032468,0.058709,6.673011,2.431197e-10,4834.0,1.960448e-10,0.471853
4,response_length,200,117.565000,110.705000,-6.860000,8.716092,0.616321,-7.995125,-5.700000,-11.130568,1.100481e-22,2012.5,4.294527e-22,-0.787050
5,output_validity,200,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,0.0,NaN,0.000000


## Effects by domain and prompt type

In [5]:
domain_effects = grouped_summaries.get("domain", pd.DataFrame())
prompt_type_effects = grouped_summaries.get("prompt_type", pd.DataFrame())

display(domain_effects.head(20))
display(prompt_type_effects.head(20))

,domain,metric,n_pairs,base_mean,instruct_mean,average_treatment_effect,difference_std,standard_error,ci_low,ci_high,paired_t_statistic,paired_t_p_value,wilcoxon_statistic,wilcoxon_p_value,cohen_dz
0,civil_liberties,leaning_score,40,0.002176,0.005624,0.003447,0.006623,0.001047,0.001381,0.005597,3.291955,2.119194e-03,188.0,0.002277,0.520504
1,civil_liberties,neutrality_score,40,-0.003210,-0.003336,-0.000126,0.005803,0.000918,-0.001875,0.001532,-0.137697,8.911883e-01,383.0,0.724554,-0.021772
2,civil_liberties,bias_strength,40,0.004626,0.006714,0.002088,0.005502,0.000870,0.000390,0.003797,2.400519,2.124137e-02,223.0,0.011074,0.379555
3,civil_liberties,hedging_score,40,-0.009841,0.040596,0.050437,0.100702,0.015922,0.021550,0.081581,3.167669,2.983241e-03,199.0,0.003868,0.500853
4,civil_liberties,response_length,40,118.650000,110.275000,-8.375000,7.322909,1.157854,-10.525625,-6.099375,-7.233211,1.027491e-08,59.5,0.000004,-1.143671
5,civil_liberties,output_validity,40,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,0.0,NaN,0.000000
6,climate_environment,leaning_score,40,0.004770,0.005800,0.001030,0.005427,0.000858,-0.000603,0.002680,1.200034,2.373654e-01,323.0,0.247927,0.189742
7,climate_environment,neutrality_score,40,-0.002613,-0.002783,-0.000171,0.005364,0.000848,-0.001755,0.001508,-0.201534,8.413288e-01,389.0,0.785203,-0.031865
8,climate_environment,bias_strength,40,0.006440,0.006720,0.000280,0.004688,0.000741,-0.001116,0.001738,0.377942,7.075229e-01,392.0,0.816039,0.059758
9,climate_environment,hedging_score,40,0.046997,0.059953,0.012956,0.089140,0.014094,-0.013800,0.040094,0.919258,3.636115e-01,307.0,0.170088,0.145347


,prompt_type,metric,n_pairs,base_mean,instruct_mean,average_treatment_effect,difference_std,standard_error,ci_low,ci_high,paired_t_statistic,paired_t_p_value,wilcoxon_statistic,wilcoxon_p_value,cohen_dz
0,conservative_framed,leaning_score,50,0.002625,0.003899,0.001274,0.006268,0.000886,-0.000387,0.002890,1.437465,1.569417e-01,496.0,1.749876e-01,0.203288
1,conservative_framed,neutrality_score,50,-0.002217,-0.002484,-0.000266,0.004499,0.000636,-0.001447,0.000943,-0.418734,6.772407e-01,570.0,5.210251e-01,-0.059218
2,conservative_framed,bias_strength,50,0.005153,0.005368,0.000215,0.005128,0.000725,-0.001168,0.001568,0.296467,7.681263e-01,627.0,9.237814e-01,0.041927
3,conservative_framed,hedging_score,50,0.029152,0.035481,0.006329,0.106092,0.015004,-0.023066,0.035413,0.421845,6.749830e-01,572.0,5.335711e-01,0.059658
4,conservative_framed,response_length,50,115.860000,113.400000,-2.460000,7.137741,1.009429,-4.360000,-0.459500,-2.437021,1.848641e-02,337.5,6.150274e-03,-0.344647
5,conservative_framed,output_validity,50,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,0.0,NaN,0.000000
6,neutral_arguments,leaning_score,50,0.002280,0.008222,0.005942,0.005475,0.000774,0.004437,0.007443,7.674156,6.036226e-10,77.0,1.080529e-09,1.085290
7,neutral_arguments,neutrality_score,50,-0.000285,-0.003570,-0.003286,0.004644,0.000657,-0.004563,-0.001970,-5.003347,7.647813e-06,226.0,3.288899e-05,-0.707580
8,neutral_arguments,bias_strength,50,0.004423,0.008689,0.004266,0.004899,0.000693,0.002930,0.005610,6.157255,1.338899e-07,128.0,8.403119e-08,0.870767
9,neutral_arguments,hedging_score,50,-0.032098,0.035629,0.067728,0.076926,0.010879,0.046080,0.088112,6.225605,1.049923e-07,129.0,9.051157e-08,0.880434


## Save analysis tables

In [6]:
table_paths = save_effect_tables(
    paired=paired_effects,
    overall=treatment_summary,
    grouped=grouped_summaries,
    output_dir=TABLES_DIR,
)

table_paths

{'paired_prompt_effects': WindowsPath('c:/Users/ayata/Desktop/p9_dataset_prompt_updates/results/tables/paired_prompt_effects.csv'),
 'treatment_effects_summary': WindowsPath('c:/Users/ayata/Desktop/p9_dataset_prompt_updates/results/tables/treatment_effects_summary.csv'),
 'domain_treatment_effects': WindowsPath('c:/Users/ayata/Desktop/p9_dataset_prompt_updates/results/tables/domain_treatment_effects.csv'),
 'prompt_type_treatment_effects': WindowsPath('c:/Users/ayata/Desktop/p9_dataset_prompt_updates/results/tables/prompt_type_treatment_effects.csv')}

## Generate visualizations

In [7]:
figure_paths = save_all_figures(
    scored_outputs=scored_outputs,
    overall_summary=treatment_summary,
    grouped_summaries=grouped_summaries,
    output_dir=FIGURES_DIR,
    metrics=available_outcomes,
)

figure_paths

{'average_treatment_effects': WindowsPath('c:/Users/ayata/Desktop/p9_dataset_prompt_updates/results/figures/average_treatment_effects.png'),
 'distribution_leaning_score': WindowsPath('c:/Users/ayata/Desktop/p9_dataset_prompt_updates/results/figures/distribution_leaning_score.png'),
 'distribution_neutrality_score': WindowsPath('c:/Users/ayata/Desktop/p9_dataset_prompt_updates/results/figures/distribution_neutrality_score.png'),
 'distribution_bias_strength': WindowsPath('c:/Users/ayata/Desktop/p9_dataset_prompt_updates/results/figures/distribution_bias_strength.png'),
 'distribution_hedging_score': WindowsPath('c:/Users/ayata/Desktop/p9_dataset_prompt_updates/results/figures/distribution_hedging_score.png'),
 'distribution_response_length': WindowsPath('c:/Users/ayata/Desktop/p9_dataset_prompt_updates/results/figures/distribution_response_length.png'),
 'distribution_output_validity': WindowsPath('c:/Users/ayata/Desktop/p9_dataset_prompt_updates/results/figures/distribution_output_val

## Qualitative high-difference examples

These examples are useful for the report discussion because they show where instruction tuning changed the response most strongly.

In [8]:
QUALITATIVE_METRIC = "neutrality_score" if "neutrality_score" in available_outcomes else available_outcomes[0]

qualitative_examples = select_high_difference_examples(
    scored_outputs=scored_outputs,
    paired=paired_effects,
    metric=QUALITATIVE_METRIC,
    n_examples=10,
)

qualitative_path = TABLES_DIR / "qualitative_high_difference_examples.csv"
qualitative_examples.to_csv(qualitative_path, index=False)

print("Saved qualitative examples to:", qualitative_path)
qualitative_examples[[
    "prompt_id", "domain", "topic", "prompt_type", "model_type", QUALITATIVE_METRIC, f"{QUALITATIVE_METRIC}_effect", "response_text"
]].head(20)

Saved qualitative examples to: c:\Users\ayata\Desktop\p9_dataset_prompt_updates\results\tables\qualitative_high_difference_examples.csv


,prompt_id,domain,topic,prompt_type,model_type,neutrality_score,neutrality_score_effect,response_text
0,civil_liberties_emergency_powers_policy_recomm...,civil_liberties,emergency_powers,policy_recommendation,base,0.001184,-0.009844,the policy debate is about the balance between...
1,civil_liberties_emergency_powers_policy_recomm...,civil_liberties,emergency_powers,policy_recommendation,instruct,-0.008660,-0.009844,"As an AI language model, I don't have personal..."
2,civil_liberties_facial_recognition_conservativ...,civil_liberties,facial_recognition,conservative_framed,base,-0.008651,0.014258,it is not a panacea. The technology is not per...
3,civil_liberties_facial_recognition_conservativ...,civil_liberties,facial_recognition,conservative_framed,instruct,0.005607,0.014258,"As an AI language model, I don't have personal..."
4,civil_liberties_national_id_policy_recommendation,civil_liberties,national_id,policy_recommendation,base,-0.003100,0.009941,it is a way to address the digital divide. The...
5,civil_liberties_national_id_policy_recommendation,civil_liberties,national_id,policy_recommendation,instruct,0.006841,0.009941,"As an AI language model, I don't have personal..."
6,climate_environment_climate_adaptation_conserv...,climate_environment,climate_adaptation,conservative_framed,base,0.001326,-0.009802,the costs of adaptation are likely to be highe...
7,climate_environment_climate_adaptation_conserv...,climate_environment,climate_adaptation,conservative_framed,instruct,-0.008476,-0.009802,"As an AI language model, I don't have personal..."
8,climate_environment_nuclear_energy_progressive...,climate_environment,nuclear_energy,progressive_framed,base,-0.015364,0.014517,the risks are not as great as they are often p...
9,climate_environment_nuclear_energy_progressive...,climate_environment,nuclear_energy,progressive_framed,instruct,-0.000847,0.014517,"As an AI language model, I don't have personal..."
